In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('../Data/2025_full_data.csv')

In [ ]:
df.head()


In [ ]:
df.columns

In [ ]:
df.isna().sum()/len(df) * 100

In [ ]:
cols_to_drop = [
    'mfr_dt', 'init_fda_dt', 'fda_dt', 
    'auth_num',
    'lit_ref',
    'age_grp',
    'rpsr_cod_rpsr',
    'primaryid',
    'caseid', 'caseid_drug', 'caseid_reac', 'caseid_ther', 'caseid_outc', 'caseid_rpsr', 'caseid_indi',
    'mfr_num',
    'to_mfr',
    'reporter_country',
    'drug_seq_drug','dsg_drug_seq_ther','indi_drug_seq_indi',
    'lot_num_drug',
    'start_dt_ther',
    'end_dt_ther',
    'dur_ther','dur_cod_ther'

]

df = df.drop(columns=cols_to_drop)

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
def convert_to_kg(row):
    if row['wt_cod'] == 'LBS':
        return row['wt'] * 0.453592
    else:  # kg
        return row['wt']

df['wt'] = df.apply(convert_to_kg, axis=1)

In [ ]:
df.columns

In [ ]:
df['wt']

In [ ]:
df['wt_cod'].value_counts()

In [ ]:
df = df.drop(columns=['wt_cod'])

In [ ]:
df.isna().sum()/len(df) * 100

In [ ]:
df["age"].median()

df["age"].mean()

In [ ]:
df['age_cod'].value_counts()

In [ ]:
def convert_to_years(row):
    code = row['age_cod'].strip().lower()

    if code == 'yr':
        return row['age']
    elif code == 'dec':
        return row['age'] * 10
    elif code == 'mon':
        return row['age'] / 12
    elif code == 'wk':
        return row['age'] / 52
    elif code == 'dy':
        return row['age'] / 365
    elif code == 'hr':
        return row['age'] / (365 * 24)
    else:
        return None
df['age_cod'] = df['age_cod'].fillna('').astype(str)
df['age'] = df.apply(convert_to_years, axis=1)

In [ ]:
df.drop(columns=['age_cod'], inplace=True)

In [ ]:
df.columns


In [ ]:
df.isna().sum()/len(df) * 100

In [ ]:
print(df['age'].median())
print(df['age'].mean())

In [ ]:
df['age']=df['age'].fillna(df['age'].mean())

In [ ]:
df.isna().sum()/len(df) * 100

In [ ]:
df['sex'].value_counts()

In [ ]:
df['sex'] = df['sex'].replace('UNK', None)

In [ ]:
df.isna().sum()/len(df) * 100

In [ ]:
df['sex']=df['sex'].fillna(df['sex'].mode()[0])

In [ ]:
df['occp_cod'].value_counts()

In [ ]:
df['occp_cod']=df['occp_cod'].fillna(df['occp_cod'].mode()[0])

In [ ]:
df.isna().sum()/len(df) * 100

In [ ]:
df['occr_country'].value_counts()

In [ ]:
df['occr_country']=df['occr_country'].fillna(df['occr_country'].mode()[0])

In [ ]:
df['route_drug']

In [ ]:
# # 1) Remove noise words and extra commas/spaces
s = (
    df["route_drug"]
    .astype("string")
    .str.replace(r"(?i)\b(unknown|nan|none)\b", "", regex=True)
    .str.replace(r"\s*,\s*", ", ", regex=True)
    .str.replace(r"^,\s*|\s*,$", "", regex=True)
    .str.replace(r"(,\s*){2,}", ", ", regex=True)
    .str.strip()
    .replace("", pd.NA)
)

# 2) Fill missing with the most common route
fill_value = s.mode().iat[0] if not s.mode().empty else "Unknown"
df["route_drug"] = s.fillna(fill_value)

# 3) Quick check
df["route_drug"].value_counts().head(20)

In [ ]:
df['route_drug']

In [ ]:
df['dose_vbm_drug']

In [ ]:
# 1) Remove noise words
s_dose = df["dose_vbm_drug"].astype("string")
s_dose = s_dose.str.replace(r"(?i)\b(unknown|nan|none|unk|na)\b", "", regex=True)

# 2) Aggressively clean up stray commas and spaces
s_dose = (
    s_dose
    .str.replace(r"[\s,]+", " ", regex=True) # Temporarily turn all comma/space blocks into single spaces
    # Alternatively, if you WANT to keep legitimate commas between multiple doses:
    # .str.replace(r"\s*,\s*", ",", regex=True)
    # .str.replace(r",+", ",", regex=True)
    # .str.replace(r"^[\s,]+|[\s,]+$", "", regex=True)
)

# For better separation, let's keep commas but clean them properly:
s_dose = df["dose_vbm_drug"].astype("string")
s_dose = s_dose.str.replace(r"(?i)\b(unknown|nan|none|unk|na)\b", "", regex=True)
s_dose = s_dose.str.replace(r"(\s*,\s*)+", ", ", regex=True) # Merge multiple commas into one ", " 
s_dose = s_dose.str.replace(r"^[\s,]+|[\s,]+$", "", regex=True) # Strip leading and trailing commas/spaces
s_dose = s_dose.replace("", pd.NA)


# 3) Fill missing with the most common dose
df["dose_vbm_drug"] = s_dose.fillna(s_dose.mode().iat[0] if not s_dose.mode().empty else "Unknown")

# 4) Quick check
df["dose_vbm_drug"].value_counts()

In [ ]:
df["dose_vbm_drug"].head(20)

In [ ]:
df.isna().sum()/len(df) * 100

In [ ]:
# 1) Clean noise words from cum_dose_chr_drug
s_cum_dose = df["cum_dose_chr_drug"].astype("string")
s_cum_dose = s_cum_dose.str.replace(r"(?i)\b(unknown|nan|none|unk|na)\b", "", regex=True)

# 2) Clean up commas and spaces
s_cum_dose = s_cum_dose.str.replace(r"(\s*,\s*)+", ", ", regex=True) # Merge multiple commas
s_cum_dose = s_cum_dose.str.replace(r"^[\s,]+|[\s,]+$", "", regex=True) # Strip leading/trailing
s_cum_dose = s_cum_dose.replace("", pd.NA)

# 3) Check if we can safely convert it to numeric (if it's purely numbers)
# We will use pd.to_numeric with errors='coerce' to see if it produces mostly valid numbers
numeric_cum_dose = pd.to_numeric(s_cum_dose, errors='coerce')

# If more than 50% of the non-null data is numeric, we treat it as a numeric column 
if numeric_cum_dose.notna().sum() > (s_cum_dose.notna().sum() * 0.5):
    # It's mostly numeric! Fill missing with the median.
    median_val = numeric_cum_dose.median() if not pd.isna(numeric_cum_dose.median()) else 0
    df["cum_dose_chr_drug"] = numeric_cum_dose.fillna(median_val)
else:
    # It's mostly text/categorical. Fill missing with the mode.
    fill_val = s_cum_dose.mode().iat[0] if not s_cum_dose.mode().empty else "Unknown"
    df["cum_dose_chr_drug"] = s_cum_dose.fillna(fill_val)

# 4) Quick check
df["cum_dose_chr_drug"].value_counts().head(20)

In [ ]:
# 1) Standardize and clean noise words from cum_dose_unit_drug
s_unit = df["cum_dose_unit_drug"].astype("string").str.upper() # Standardize to uppercase for units (e.g., MG, ML)
s_unit = s_unit.str.replace(r"(?i)\b(UNKNOWN|NAN|NONE|UNK|NA)\b", "", regex=True)

# 2) Clean up commas and spaces
s_unit = s_unit.str.replace(r"(\s*,\s*)+", ", ", regex=True) # Merge multiple commas
s_unit = s_unit.str.replace(r"^[\s,]+|[\s,]+$", "", regex=True) # Strip leading/trailing
s_unit = s_unit.replace("", pd.NA)

# 3) Fill missing with the most common unit
fill_val_unit = s_unit.mode().iat[0] if not s_unit.mode().empty else "Unknown"
df["cum_dose_unit_drug"] = s_unit.fillna(fill_val_unit)

# 4) Quick check
df["cum_dose_unit_drug"]

In [ ]:
df["cum_dose_unit_drug"].head(20)

In [ ]:
df["cum_dose_chr_drug"].head(20)

In [ ]:
df.isna().sum()/len(df) * 100

In [ ]:
df['dechal_drug'].value_counts().head(20)

In [ ]:
def encode_dechal(val):
    if pd.isna(val):
        return -1
    
    # Convert to uppercase string to be safe
    val_str = str(val).upper()
    
    # If the patient/report had ANY drug that was dechallenged (Y), consider it 1
    if 'Y' in val_str:
        return 1
    # If no drug was affirmatively dechallenged, but at least one wasn't (N), consider it 0
    elif 'N' in val_str:
        return 0
    # Otherwise, it's Unknown (U), Does not apply (D), or missing
    else:
        return -1

# Apply the custom parser across the entire column
df['dechal_drug'] = df['dechal_drug'].apply(encode_dechal)

# Quick check
df['dechal_drug'].value_counts()

In [ ]:
df['rechal_drug'].value_counts()

In [ ]:
def encode_rechal(val):
    if pd.isna(val):
        return -1
    
    # Convert to uppercase string to be safe
    val_str = str(val).upper()
    
    # If the patient/report had ANY drug that was rechallenged (Y), consider it 1
    if 'Y' in val_str:
        return 1
    # If no drug was affirmatively rechallenged, but at least one wasn't (N), consider it 0
    elif 'N' in val_str:
        return 0
    # Otherwise, it's Unknown (U), Does not apply (D), or missing
    else:
        return -1

# Apply the custom parser across the entire column
df['rechal_drug'] = df['rechal_drug'].apply(encode_rechal)

# Quick check
df['rechal_drug'].value_counts()

In [ ]:
df.isna().sum()/len(df) * 100

In [ ]:
# Drop the original 'exp_dt_drug' column from the DataFrame
df.drop(columns=['exp_dt_drug'], inplace=True)

# Verify it was dropped
'exp_dt_drug' in df.columns

In [ ]:
# Drop the extra 'exp_year' and 'exp_month' columns we created
df.drop(columns=['exp_year', 'exp_month'], inplace=True, errors='ignore')

# Quick check of what columns remain
df.columns

In [ ]:
# Drop the 'nda_num_drug' column as it's just an identifier and likely noise for modeling
df.drop(columns=['nda_num_drug'], inplace=True, errors='ignore')

# Verify it was dropped
'nda_num_drug' in df.columns

In [ ]:
df.isna().sum()/len(df) * 100

In [ ]:
# 1) Convert to string
s_amt = df['dose_amt_drug'].astype(str)

# 2) Remove noise words (like 'nan', 'unknown', 'none', etc.)
s_amt = s_amt.str.replace(r"(?i)\b(unknown|nan|none|unk|na)\b", "", regex=True)

# 3) Clean up stray commas and spaces
s_amt = s_amt.str.replace(r"(\s*,\s*)+", ", ", regex=True)
s_amt = s_amt.str.replace(r"^[\s,]+|[\s,]+$", "", regex=True)

# 4) Replace empty strings with actual missing values
s_amt = s_amt.replace("", pd.NA)

# 5) Determine how to handle the numeric conversion vs strings
# Many rows have comma-separated numbers like "600.0, 300.0". 
# For modeling, a common strategy is to extract the median or max of these lists, 
# or just take the first reported valid number if order matters.
def get_numeric_representation(val):
    if pd.isna(val):
        return np.nan
    try:
        # Split by comma, convert to float (ignoring any completely broken strings)
        nums = [float(x.strip()) for x in val.split(',') if x.strip()]
        # Return median dosage amount if list is valid, else nan
        return np.median(nums) if nums else np.nan
    except:
        return np.nan

# Create numeric series
numeric_amt = s_amt.apply(get_numeric_representation)

# 6) Fill the remaining missing numbers with the overall median
median_amt = numeric_amt.median()
df['dose_amt_drug'] = numeric_amt.fillna(median_amt)

# 7) Check the clean result
df['dose_amt_drug'].value_counts().head(20)

In [ ]:
# 1) Standardize and clean noise words from dose_unit_drug
s_unit_amt = df["dose_unit_drug"].astype("string").str.upper() # Standardize to uppercase
s_unit_amt = s_unit_amt.str.replace(r"(?i)\b(UNKNOWN|NAN|NONE|UNK|NA)\b", "", regex=True)

# 2) Clean up commas and spaces
s_unit_amt = s_unit_amt.str.replace(r"(\s*,\s*)+", ", ", regex=True) # Merge multiple commas
s_unit_amt = s_unit_amt.str.replace(r"^[\s,]+|[\s,]+$", "", regex=True) # Strip leading/trailing

# 3) Deduplicate repeating units (e.g., "MG, MG, MG" -> "MG") as multiple drugs often share the same unit
def deduplicate_units(val):
    if pd.isna(val) or val.strip() == "":
        return pd.NA
    parts = [p.strip() for p in val.split(',') if p.strip()]
    unique_parts = list(dict.fromkeys(parts))
    return ", ".join(unique_parts)

s_unit_amt = s_unit_amt.apply(deduplicate_units)

# 4) Fill missing with the most common unit
fill_val_unit_amt = s_unit_amt.mode().iat[0] if not s_unit_amt.mode().empty else "Unknown"
df["dose_unit_drug"] = s_unit_amt.fillna(fill_val_unit_amt)

# 5) Quick check
df["dose_unit_drug"].value_counts().head(20)

In [ ]:
# 1) Standardize and clean noise words from dose_form_drug
s_form = df["dose_form_drug"].astype("string").str.upper() # Standardize to uppercase
s_form = s_form.str.replace(r"(?i)\b(UNKNOWN|NAN|NONE|UNK|NA)\b", "", regex=True)

# 2) Clean up commas and spaces
s_form = s_form.str.replace(r"(\s*,\s*)+", ", ", regex=True) # Merge multiple commas
s_form = s_form.str.replace(r"^[\s,]+|[\s,]+$", "", regex=True) # Strip leading/trailing

# 3) Deduplicate repeating forms (e.g., "TABLET, TABLET" -> "TABLET")
# We can reuse the deduplicate_units function defined earlier!
s_form = s_form.apply(deduplicate_units)

# 4) Fill missing with the most common form
fill_val_form = s_form.mode().iat[0] if not s_form.mode().empty else "Unknown"
df["dose_form_drug"] = s_form.fillna(fill_val_form)

# 5) Quick check
df["dose_form_drug"].value_counts().head(20)

In [ ]:
df.isna().sum()/len(df) * 100

In [ ]:
# 1) Standardize and clean noise words from dose_freq_drug
s_freq = df["dose_freq_drug"].astype("string").str.upper() # Standardize to uppercase
s_freq = s_freq.str.replace(r"(?i)\b(UNKNOWN|NAN|NONE|UNK|NA)\b", "", regex=True)

# 2) Clean up commas and spaces
s_freq = s_freq.str.replace(r"(\s*,\s*)+", ", ", regex=True) # Merge multiple commas
s_freq = s_freq.str.replace(r"^[\s,]+|[\s,]+$", "", regex=True) # Strip leading/trailing

# 3) Deduplicate repeating frequencies (e.g., "BID, BID" -> "BID")
s_freq = s_freq.apply(deduplicate_units)

# 4) Fill missing with the most common frequency
fill_val_freq = s_freq.mode().iat[0] if not s_freq.mode().empty else "Unknown"
df["dose_freq_drug"] = s_freq.fillna(fill_val_freq)

# 5) Quick check
df["dose_freq_drug"].value_counts().head(20)

In [ ]:
df['drug_rec_act_reac'].value_counts()

In [ ]:
# 1) Standardize and clean noise words from drug_rec_act_reac
s_reac = df['drug_rec_act_reac'].astype("string").str.title() # Standardize casing (e.g., Headache)
s_reac = s_reac.str.replace(r"(?i)\b(UNKNOWN|NAN|NONE|UNK|NA)\b", "", regex=True)

# 2) Clean up commas and spaces
s_reac = s_reac.str.replace(r"(\s*,\s*)+", ", ", regex=True) # Merge multiple commas
s_reac = s_reac.str.replace(r"^[\s,]+|[\s,]+$", "", regex=True) # Strip leading/trailing

# 3) Deduplicate repeating reactions for the same case (e.g., "Headache, Headache" -> "Headache")
s_reac = s_reac.apply(deduplicate_units)

# 4) Fill missing with a placeholder ("Unknown" or "No Reaction Reported")
s_reac = s_reac.replace("", pd.NA)
fill_val_reac = s_reac.mode().iat[0] if not s_reac.mode().empty else "Unknown"
df["drug_rec_act_reac"] = s_reac.fillna(fill_val_reac)

# 5) Quick check
df["drug_rec_act_reac"].value_counts().head(20)

In [ ]:
df.isna().sum()/len(df) * 100

In [ ]:
# 1) Standardize and clean noise words from outc_cod_outc
s_outc = df['outc_cod_outc'].astype("string").str.upper()
s_outc = s_outc.str.replace(r"(?i)\b(UNKNOWN|NAN|NONE|UNK|NA)\b", "", regex=True)

# 2) Clean up commas and spaces
s_outc = s_outc.str.replace(r"(\s*,\s*)+", ", ", regex=True)
s_outc = s_outc.str.replace(r"^[\s,]+|[\s,]+$", "", regex=True)

# 3) Deduplicate AND SORT the outcomes
# Order of outcomes doesn't matter, so "HO, OT" is identical to "OT, HO". 
# Sorting ensures we don't accidentally create separate categories for the exact same patient outcomes.
def deduplicate_and_sort(val):
    if pd.isna(val) or str(val).strip() == "":
        return pd.NA
    parts = [p.strip() for p in str(val).split(',') if p.strip()]
    unique_sorted = sorted(list(set(parts)))
    return ", ".join(unique_sorted)

s_outc = s_outc.apply(deduplicate_and_sort)

# 4) Handle missing values safely
# Because outcomes are typically the target variable, making assumptions by filling with the mode 
# might introduce bias. Pinning it definitively as "Unknown" is the safest classification.
s_outc = s_outc.replace("", pd.NA)
df["outc_cod_outc"] = s_outc.fillna("Unknown")

# 5) Quick check
df["outc_cod_outc"].value_counts().head(20)

In [ ]:
# 1) Standardize and clean noise words from indi_pt_indi (Indications)
s_indi = df['indi_pt_indi'].astype("string").str.title() # Use title case for standard disease/medical names

# Some specific FAERS-centric noise in indication fields BEFORE standard regex:
s_indi = s_indi.str.replace(r"(?i)\bProduct Used For Unknown Indication\b", "", regex=True)
# Also remove occurrences where removing 'Unknown' left weird artifacts
s_indi = s_indi.str.replace(r"(?i)\bProduct Used For\s*Indication\b", "", regex=True) 

# Remove standard nan/none/unk/unknown
s_indi = s_indi.str.replace(r"(?i)\b(UNKNOWN|NAN|NONE|UNK|NA)\b", "", regex=True)

# 2) Clean up commas and spaces
s_indi = s_indi.str.replace(r"(\s*,\s*)+", ", ", regex=True) # Merge multiple commas
s_indi = s_indi.str.replace(r"^[\s,]+|[\s,]+$", "", regex=True) # Strip leading/trailing

# 3) Deduplicate AND SORT the indications per patient
# A patient taking drugs for "Asthma, Hypertension" is the same as "Hypertension, Asthma".
# We use the deduplicate_and_sort function we wrote for the outcomes:
s_indi = s_indi.apply(deduplicate_and_sort)

# 4) Handle specific missing values 
s_indi = s_indi.replace("", pd.NA)
fill_val_indi = s_indi.mode().iat[0] if not s_indi.mode().empty else "Unknown"
df["indi_pt_indi"] = s_indi.fillna(fill_val_indi)

# 5) Quick check
df["indi_pt_indi"].value_counts().head(20)

In [ ]:
df['event_dt']

In [ ]:
df.isna().sum()/len(df) * 100

In [ ]:
df['wt']

In [ ]:
df = df.drop(columns=['wt'])

In [ ]:
df.isna().sum()/len(df) * 100

In [ ]:
df['event_dt']

In [ ]:
len(df)

In [ ]:
df.drop_duplicates(inplace=True)